In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [4]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,2,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,2,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,2,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,2,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,9,0.0
26996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,9,0.0
26997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,9,0.0
26998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,9,0.0


In [5]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [6]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [7]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  413159.272588
                                  2                  245276.294149
                                  3                  312863.941400
                                  4                  249072.466042
                                  5                  174991.786434
intervention  maternal_disorders  1                  413159.272588
                                  2                  245276.294149
                                  3                  312863.941400
                                  4                  249072.466042
                                  5                  174991.786434
zero          maternal_disorders  1                  422333.247189
                                  2                  250562.470712
                                  3                  319137.428004
                                  4                  255097.746825
            

In [8]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,2,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,2,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,2,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,2,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
94495,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,9,0.0
94496,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,9,0.0
94497,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,9,0.0
94498,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,9,0.0


In [9]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [10]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   96876.056921
                                  2                   76056.436913
                                  3                   60833.604687
                                  4                   52288.312222
                                  5                   40608.233098
              maternal_disorders  1                     278.284931
                                  2                     146.599510
                                  3                     170.482749
                                  4                     171.631578
                                  5                     118.299038
intervention  anemia              1                   96876.056921
                                  2                   76056.436913
                                  3                   60833.604687
                                  4                   52288.312222
            

In [11]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   96876.056921
                                  2                   76056.436913
                                  3                   60833.604687
                                  4                   52288.312222
                                  5                   40608.233098
              maternal_disorders  1                  413437.557518
                                  2                  245422.893659
                                  3                  313034.424149
                                  4                  249244.097620
                                  5                  175110.085471
intervention  anemia              1                   96876.056921
                                  2                   76056.436913
                                  3                   60833.604687
                                  4                   52288.312222
            

In [12]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [13]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# TODO: Determine whether it's expected that the child_scenario column
# always contains only 'baseline'. If there's more than one scenario in
# this column, then results from different child scenarios would get
# added together in the call to aggregate_by_cause_and_scenario below.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # TODO: Explain why this branch exists. For example, is this for
    # processing the Ethiopia results, where no Vivarium sims were run,
    # so the corresponding DALYs should just be 0?
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,636447.934300
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,484929.084023
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,480467.999335
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,450231.535335
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,458923.455912
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,75276.583283
1196,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,62943.046988
1197,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,54762.885237
1198,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,33571.282938


In [14]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.710643e+07
                      2                  1.428245e+07
                      3                  1.289558e+07
                      4                  1.172546e+07
                      5                  1.163456e+07
intervention  lbwsg   1                  1.710643e+07
                      2                  1.428245e+07
                      3                  1.289558e+07
                      4                  1.172546e+07
                      5                  1.163456e+07
zero          lbwsg   1                  1.714974e+07
                      2                  1.428244e+07
                      3                  1.294756e+07
                      4                  1.172546e+07
                      5                  1.164323e+07
Name: value, dtype: float64

In [15]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1210.157219,zero
1,Female,0.0,0.019178,2,919.369830,zero
2,Female,0.0,0.019178,3,814.908884,zero
3,Female,0.0,0.019178,4,677.670154,zero
4,Female,0.0,0.019178,5,474.324531,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,67.051648,intervention
746,Male,95.0,125.000000,2,62.475408,intervention
747,Male,95.0,125.000000,3,64.053561,intervention
748,Male,95.0,125.000000,4,60.140287,intervention


In [16]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.762340e+06
                      2                  1.722092e+06
                      3                  1.763970e+06
                      4                  1.682876e+06
                      5                  1.531640e+06
intervention  anemia  1                  1.762340e+06
                      2                  1.722092e+06
                      3                  1.763970e+06
                      4                  1.682876e+06
                      5                  1.531640e+06
zero          anemia  1                  1.883992e+06
                      2                  1.840719e+06
                      3                  1.871022e+06
                      4                  1.773993e+06
                      5                  1.579529e+06
Name: value, dtype: float64

In [17]:
scenarios[1]

'zero'

In [18]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-515376.93856028095

In [19]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  629933.833088
                      2                  514172.099324
                      3                  474467.273805
                      4                  407150.123198
                      5                  318818.954021
intervention  anemia  1                  629933.833088
                      2                  514172.099324
                      3                  474467.273805
                      4                  407150.123198
                      5                  318818.954021
zero          anemia  1                  665234.725724
                      2                  543849.938688
                      3                  499059.693990
                      4                  425733.873127
                      5                  327265.912900
Name: value, dtype: float64

In [20]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-116601.86099258438

In [21]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.291040e+06
                      2                  3.886437e+06
                      3                  3.836892e+06
                      4                  3.536131e+06
                      5                  3.144628e+06
intervention  anemia  1                  4.291040e+06
                      2                  3.886437e+06
                      3                  3.836892e+06
                      4                  3.536131e+06
                      5                  3.144628e+06
zero          anemia  1                  4.589856e+06
                      2                  4.153883e+06
                      3                  4.069819e+06
                      4                  3.726689e+06
                      5                  3.240978e+06
Name: value, dtype: float64

In [22]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  363252.788867
                      2                  320895.928809
                      3                  288089.125399
                      4                  271653.847670
                      5                  231356.178732
baseline      ntd     1                  337639.680947
                      2                  301593.696614
                      3                  273439.169930
                      4                  259642.130519
                      5                  226770.777053
intervention  ntd     1                  191552.254875
                      2                  184114.187482
                      3                  178725.167952
                      4                  178459.622114
                      5                  189893.352485
Name: value, dtype: float64

In [23]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.387916e+06
                                  2                  3.962494e+06
                                  3                  3.897726e+06
                                  4                  3.588419e+06
                                  5                  3.185237e+06
              lbwsg               1                  1.710643e+07
                                  2                  1.428245e+07
                                  3                  1.289558e+07
                                  4                  1.172546e+07
                                  5                  1.163456e+07
              maternal_disorders  1                  4.134376e+05
                                  2                  2.454229e+05
                                  3                  3.130344e+05
                                  4                  2.492441e+05
                          

In [24]:
import pathlib

In [25]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)